# Optimization Algorithms
---
> Simple English | Interview Ready | Why Adam? SGD vs Adam?

## What is an Optimizer?
- Updates weights to minimize the loss function
- Gradient Descent is the base idea; optimizers are improved versions

## All Optimizers
| Optimizer | Idea | Verdict |
|---|---|---|
| Batch GD | Use all data | Stable but slow |
| SGD | 1 sample/step | Fast but noisy |
| Mini-Batch GD | Small batch | Best balance (default) |
| Momentum | Add velocity | Faster, escapes local min |
| RMSProp | Adaptive LR per param | Good for RNNs |
| Adam | Momentum + RMSProp | Best all-rounder |

## Adam Formula
```
m = b1*m + (1-b1)*g         (momentum term)
v = b2*v + (1-b2)*g*g       (RMSProp term)
m_hat = m/(1-b1^t)          (bias correction)
v_hat = v/(1-b2^t)
w = w - lr * m_hat / (sqrt(v_hat) + eps)

Default: lr=0.001, b1=0.9, b2=0.999
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
noisy_grad = lambda w: 2*(w-3) + np.random.randn()*2
lr=0.1; steps=25

# 1. Batch GD (stable)
w=0.0; path_b=[w]
for _ in range(steps): w -= lr*2*(w-3); path_b.append(w)

# 2. SGD (noisy)
w=0.0; path_s=[w]
for _ in range(steps): w -= lr*noisy_grad(w); path_s.append(w)

# 3. Momentum
w=0.0; v=0; path_m=[w]
for _ in range(steps):
    g=noisy_grad(w); v=0.9*v+0.1*g; w -= lr*v; path_m.append(w)

plt.figure(figsize=(10,4))
plt.plot(range(steps+1),path_b,'b-o',label='Batch GD',lw=2)
plt.plot(range(steps+1),path_s,'r-o',label='SGD (noisy)',lw=2,alpha=0.7)
plt.plot(range(steps+1),path_m,'g-o',label='Momentum',lw=2)
plt.axhline(3,color='k',linestyle='--',label='Optimal w=3')
plt.title('Batch GD vs SGD vs Momentum'); plt.legend(); plt.grid(True,alpha=0.3)
plt.xlabel('Step'); plt.ylabel('Weight'); plt.show()

In [ ]:
# Adam manual implementation
import numpy as np

np.random.seed(42)
w=0.0; lr=0.01; b1=0.9; b2=0.999; eps=1e-8; m=0; v=0
path_adam=[w]

for t in range(1,100):
    g = 2*(w-3) + np.random.randn()*0.5
    m = b1*m + (1-b1)*g
    v = b2*v + (1-b2)*g*g
    m_hat = m/(1-b1**t)
    v_hat = v/(1-b2**t)
    w -= lr * m_hat/(np.sqrt(v_hat)+eps)
    path_adam.append(w)

print(f"Adam converged to: w = {w:.4f}  (target: 3.0)")
print("Adam = Momentum (direction) + RMSProp (adaptive step size)")

In [ ]:
# Compare all optimizers in Keras
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X,y = make_classification(n_samples=1000, n_features=10, random_state=42)
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2)
sc = StandardScaler()
Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)

def build():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64,activation='relu',input_shape=(10,)),
        tf.keras.layers.Dense(32,activation='relu'),
        tf.keras.layers.Dense(1,activation='sigmoid')
    ])

opts = {'SGD (lr=0.01)':tf.keras.optimizers.SGD(0.01),
        'SGD+Momentum' :tf.keras.optimizers.SGD(0.01,momentum=0.9,nesterov=True),
        'RMSProp'      :tf.keras.optimizers.RMSprop(0.001),
        'Adam (default)':tf.keras.optimizers.Adam(0.001)}

plt.figure(figsize=(12,5))
for name,opt in opts.items():
    m=build(); m.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])
    h=m.fit(Xtr,ytr,epochs=30,batch_size=32,validation_split=0.1,verbose=0)
    plt.plot(h.history['val_accuracy'],label=name,lw=2)
plt.title('Optimizer Comparison — Val Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True,alpha=0.3); plt.show()

In [ ]:
# Learning Rate Scheduling
import tensorflow as tf

# 1. Exponential Decay
exp_decay = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01, decay_steps=100, decay_rate=0.9)
print("ExponentialDecay: LR decays by 10% every 100 steps")

# 2. Cosine Decay (popular in modern DL)
cos_decay = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.01, decay_steps=1000)
print("CosineDecay: LR follows cosine from 0.01 to 0")

# 3. ReduceLROnPlateau (most practical)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
print("ReduceLROnPlateau: halve LR when val_loss stalls for 5 epochs")
print()
print("Usage with model:")
print("  model.fit(..., callbacks=[reduce_lr])")

## Interview Questions

**Q: Why is Adam the most popular optimizer?**
> Combines Momentum (direction memory) + RMSProp (adaptive per-parameter step size). Works well with default settings (lr=0.001). Faster convergence than plain SGD. Handles sparse gradients well.

**Q: SGD vs Adam — which to use?**
> Adam: faster convergence, easier to tune, good for most NLP/general tasks.
> SGD+Momentum: sometimes better final accuracy in computer vision (ResNets trained with SGD).
> Start with Adam. Switch to SGD if overfitting.

**Q: What is Momentum?**
> Adds "velocity" from previous gradient steps. Helps escape local minima. v = 0.9*v_prev + 0.1*gradient. Reduces zigzag oscillation.

**Q: What is learning rate scheduling?**
> Start with higher LR (fast learning), reduce it over time (fine-tuning). Common: step decay, exponential decay, cosine decay, ReduceLROnPlateau.